# Enable Audio/Video visibility

Open a pad's settings and check whether the Enable Audio/Video control is
visible or absent, as specified by the supplied expectation.


## Target and expectation

Provide the pad URL and the expected result: `Absent` or `Visible`.


In [ ]:
pad_url = ""
expected_visibility = ""
screenshot_path = "../artifacts/menu.png"


## Open the pad

Wait for the editor to appear and acknowledge the initial pad-deletion-token
notice before opening the settings.


In [ ]:
from pathlib import Path

from IPython.display import Image, display
from playwright.async_api import async_playwright, expect

assert pad_url, "A pad URL is required"
assert expected_visibility in {"Absent", "Visible"}

pw = await async_playwright().start()
browser = await pw.chromium.launch(args=["--no-sandbox", "--disable-dev-shm-usage"])
context = await browser.new_context(locale="en-US")
page = await context.new_page()
await page.goto(pad_url)
await expect(page.frame_locator('iframe[name="ace_outer"]').frame_locator(
    'iframe[name="ace_inner"]').locator('body')).to_be_visible(timeout=60000)
acknowledge = page.get_by_role('button', name="I've saved it", exact=True)
await acknowledge.click()
await expect(acknowledge).to_be_hidden()


## Open settings

Open the settings menu and record the screen after it finishes appearing.
The screenshot below shows the menu used for the check.


In [ ]:
await page.locator('[data-key="settings"] button').click()
await expect(page.locator('#settings')).to_be_visible()
await expect(page.locator('#settings')).to_have_css('opacity', '1')
Path(screenshot_path).parent.mkdir(parents=True, exist_ok=True)
await page.screenshot(path=screenshot_path, full_page=True)
display(Image(filename=screenshot_path))


## Check the expected result

For `Visible`, the Enable Audio/Video control must be visible in the menu.
For `Absent`, it must not be present. A failed assertion fails this notebook;
the menu screenshot above remains available for review.


In [ ]:
print("Expected:", expected_visibility)
control = page.get_by_role('checkbox', name="Enable Audio/Video Chat", exact=True)
try:
    if expected_visibility == "Visible":
        await expect(control).to_be_visible()
    else:
        await expect(control).to_have_count(0)
    print("PASS: Enable Audio/Video is", expected_visibility)
finally:
    await browser.close()
    await pw.stop()
